# NB03 — DF40 Scaling: Competence–Calibration Coupling Across Generators

**Goal:** test whether the coupling found on FF++ (Pearson r=-0.82: calibration error rises
as detection competence falls) **holds across DF40's diverse generators** (FS / FR / EFS / FE
families spanning GANs, diffusion, swap, reenactment, editing).

Each DF40 method scored with the **FS-trained Xception** gives one (AUC, ECE_cal) point.
9 local core methods now → if the coupling holds, scale to all 40.

**Pipeline reuses:** `src/inference.py` (load detector, score) + `src/calibrate_scores.py`
(leakage-safe calibration) + `src/data_prep.py` (build_manifest_from_json).

**Lessons baked in:** import-isolation for DeepfakeBench, checkpoint path-hashing, the
inference↔calibration path switch, local-disk checkpoint loading.


## Cell 1 — Mount + paths + git creds

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, sys, glob
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
PARENT = "/content/drive/MyDrive/CDTS_Research"
DFB = f"{REPO}/external/DeepfakeBench"
# git identity (survives restarts)
import subprocess
for f in [".gitconfig", ".git-credentials"]:
    if os.path.exists(f"{PARENT}/{f}"): subprocess.run(f'cp "{PARENT}/{f}" /root/{f}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
print("REPO:", os.path.isdir(REPO))
print("DF40 core zips (local copy):")
for z in sorted(glob.glob(f"{REPO}/data/df40_core/test/*.zip")):
    print(f"  {os.path.getsize(z)/1e9:.2f}GB  {os.path.basename(z)}")
print("DF40 JSONs:", len(glob.glob(f"{REPO}/data/df40/dataset_json/*.json")))

Mounted at /content/drive
REPO: True
DF40 core zips (local copy):
  2.17GB  StyleGAN2.zip
  2.05GB  blendface.zip
  2.85GB  ddim.zip
  1.61GB  facevid2vid.zip
  1.58GB  fomm.zip
  1.39GB  inswap.zip
  15.80GB  sd2.1.zip
  2.08GB  simswap.zip
  0.24GB  styleclip.zip
DF40 JSONs: 82


## Cell 2 — DIAGNOSTIC: where do DF40 frames physically live?

Before the full pipeline, resolve the key unknown: the DF40 JSON references real (label 0)
and fake (label 1) frames — but do those paths resolve to files on disk after we unzip?
We unzip ONE method (simswap), peek at its JSON paths, and check what actually exists.


In [2]:
import json, os, glob, zipfile

REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
method = "simswap"   # a core method we have locally

# 1) peek at the JSON's frame paths (don't unzip yet)
jpath = f"{REPO}/data/df40/dataset_json/{method}_ff.json"
J = json.load(open(jpath))
top = list(J.keys())[0]                      # e.g. 'simswap_ff'
subs = list(J[top].keys())                   # e.g. ['simswap_Real','simswap_Fake']
print("JSON top key:", top, "| sub-keys:", subs)
# grab a sample frame path from a Fake and a Real entry
for sub in subs:
    for split in ['test','train','val']:
        if split in J[top][sub] and J[top][sub][split]:
            vid = list(J[top][sub][split].keys())[0]
            frames = J[top][sub][split][vid]['frames']
            print(f"  {sub}/{split}/{vid}: {len(frames)} frames")
            print(f"    sample path: {frames[0]}")
            break

# 2) look inside the zip WITHOUT extracting — what's the internal structure?
zp = f"{REPO}/data/df40_core/test/{method}.zip"
print(f"\n=== {method}.zip internal structure (first 8 entries) ===")
with zipfile.ZipFile(zp) as z:
    names = z.namelist()
    print(f"  total entries: {len(names)}")
    for n in names[:8]:
        print("   ", n)

JSON top key: simswap_ff | sub-keys: ['simswap_Real', 'simswap_Fake']
  simswap_Real/test/953: 32 frames
    sample path: deepfakes_detection_datasets/FaceForensics++/original_sequences/youtube/c23/frames/953/350.png
  simswap_Fake/test/306_278: 32 frames
    sample path: deepfakes_detection_datasets/DF40/simswap/ff/frames/306_278/328.png

=== simswap.zip internal structure (first 8 entries) ===
  total entries: 49580
    cdf/frames/id9_id0_0006/374.png
    cdf/frames/id9_id0_0006/077.png
    cdf/frames/id9_id0_0006/180.png
    cdf/frames/id9_id0_0006/154.png
    cdf/frames/id9_id0_0006/309.png
    cdf/frames/id9_id0_0006/258.png
    cdf/frames/id9_id0_0006/270.png
    cdf/frames/id9_id0_0006/245.png


**Read the diagnostic:** Compare the JSON's `sample path` to the zip's internal paths.
- If the JSON paths match the zip layout (e.g. both `simswap/ff/cdf/.../frame.png`) → unzip resolves them.
- The JSON likely has a path PREFIX (the original authors' root) we must strip/remap to our unzip location.
The next cell figures out the remap.


## Cell 3 — Unzip one method + figure out the path remap

In [3]:
import zipfile, os, glob, json
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
method = "simswap"

# unzip to LOCAL disk (fast, and frames read faster than Drive during scoring)
DF40_FRAMES = "/content/df40_frames"
os.makedirs(DF40_FRAMES, exist_ok=True)
zp = f"{REPO}/data/df40_core/test/{method}.zip"
print(f"unzipping {method}...")
with zipfile.ZipFile(zp) as z:
    z.extractall(DF40_FRAMES)
# what landed?
extracted = glob.glob(f"{DF40_FRAMES}/**/*.png", recursive=True)
print(f"extracted {len(extracted)} png frames")
if extracted:
    print("sample extracted path:", extracted[0])
    # the common root of extracted frames
    print("extraction root:", DF40_FRAMES)

unzipping simswap...
extracted 24790 png frames
sample extracted path: /content/df40_frames/cdf/frames/id30_id29_0000/308.png
extraction root: /content/df40_frames


## Cell 4 — Build manifest for the method (remap JSON paths → local frames)

Use the path remap discovered above so `frame_path` points at the unzipped local files.
We build a manifest with the same columns the scorer expects.


In [4]:
import sys, importlib.util, json, os, glob
import pandas as pd
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
method = "simswap"
DF40_FRAMES = "/content/df40_frames"

# load data_prep
spec = importlib.util.spec_from_file_location("data_prep", f"{REPO}/src/data_prep.py")
data_prep = importlib.util.module_from_spec(spec); sys.modules["data_prep"]=data_prep
spec.loader.exec_module(data_prep)

jpath = f"{REPO}/data/df40/dataset_json/{method}_ff.json"

# build manifest via the proven builder, then remap frame_path to local extraction
df = data_prep.build_manifest_from_json(f"{method}_ff", jpath, frames_root=None)
print("raw manifest rows:", len(df), "| label balance:", df['label'].value_counts().to_dict())
print("sample frame_path from manifest:", df['frame_path'].iloc[0])

# --- REMAP: the JSON paths have an author prefix; map the tail onto DF40_FRAMES ---
# We find each frame by its (video_id, filename) tail under the extraction dir.
# Build an index of extracted files: {basename_dir/file -> fullpath}
import collections
ext_index = {}
for fp in glob.glob(f"{DF40_FRAMES}/**/*.png", recursive=True):
    key = "/".join(fp.split("/")[-2:])   # videoid/frame.png
    ext_index[key] = fp

def remap(p):
    key = "/".join(p.split("/")[-2:])
    return ext_index.get(key)

df["frame_path"] = df["frame_path"].map(remap)
before = len(df)
df = df[df["frame_path"].notna()].reset_index(drop=True)
print(f"after remap: {len(df)}/{before} frames resolved to local files")
print("label balance after remap:", df['label'].value_counts().to_dict())

# if reals (label 0) are MISSING here, they live in the separate real zip — handle in Cell 5
df.to_parquet(f"/content/df40_{method}_manifest.parquet", index=False)
print(f"saved manifest for {method}")

raw manifest rows: 63290 | label balance: {1: 35818, 0: 27472}
sample frame_path from manifest: deepfakes_detection_datasets/FaceForensics++/original_sequences/youtube/c23/frames/071/277.png
after remap: 8936/63290 frames resolved to local files
label balance after remap: {1: 8936}
saved manifest for simswap


**Critical check:** does the remapped manifest have BOTH labels?
- If label 0 (real) count > 0 → reals are inside the method zip, we're good.
- If label 0 == 0 → DF40 method zips are **fake-only** (as the docs say!), and we must
  add real frames from the separate `ffpp_real.zip`. Cell 5 handles that case.


## Cell 5 — (IF NEEDED) add real frames from ffpp_real.zip

In [5]:
# DF40 docs: method folders are FAKE-ONLY; real data is shared (FF++ real).
# If Cell 4 showed label 0 == 0, run this to unzip FF++ real and attach.
import zipfile, os, glob
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
REAL_ZIP = f"{REPO}/data/df40/ffpp_real.zip"
REAL_DIR = "/content/df40_real"

if os.path.exists(REAL_ZIP) and not os.path.isdir(REAL_DIR):
    os.makedirs(REAL_DIR, exist_ok=True)
    print("unzipping FF++ real...")
    with zipfile.ZipFile(REAL_ZIP) as z:
        z.extractall(REAL_DIR)
reals = glob.glob(f"{REAL_DIR}/**/*.png", recursive=True)
print(f"real frames available: {len(reals)}")
if reals: print("sample real path:", reals[0])

unzipping FF++ real...
real frames available: 31949
sample real path: /content/df40_real/FaceForensics++/original_sequences/youtube/c23/frames/309/027.png


## Cell 6 — Score the method with FS-trained Xception

Restore DeepfakeBench on path (for the detector import), load inference.py by file path,
score the method's manifest. Same flow proven in NB02.


In [7]:
!pip -q install efficientnet_pytorch timm einops kornia simplejson 2>&1 | tail -2
print("deps installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.7 MB/s eta 0:00:00
deps installed


In [8]:
import json, os, glob, zipfile
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
method = "simswap"

# 1) peek at the JSON's frame paths
jpath = f"{REPO}/data/df40/dataset_json/{method}_ff.json"
J = json.load(open(jpath))
top = list(J.keys())[0]
subs = list(J[top].keys())
print("JSON top key:", top, "| sub-keys:", subs)
for sub in subs:
    for split in ['test','train','val']:
        if split in J[top][sub] and J[top][sub][split]:
            vid = list(J[top][sub][split].keys())[0]
            frames = J[top][sub][split][vid]['frames']
            print(f"  {sub}/{split}/{vid}: {len(frames)} frames")
            print(f"    sample path: {frames[0]}")
            break

# 2) look inside the zip without extracting
zp = f"{REPO}/data/df40_core/test/{method}.zip"
print(f"\n=== {method}.zip internal structure (first 8 entries) ===")
with zipfile.ZipFile(zp) as z:
    names = z.namelist()
    print(f"  total entries: {len(names)}")
    for n in names[:8]:
        print("   ", n)

JSON top key: simswap_ff | sub-keys: ['simswap_Real', 'simswap_Fake']
  simswap_Real/test/953: 32 frames
    sample path: deepfakes_detection_datasets/FaceForensics++/original_sequences/youtube/c23/frames/953/350.png
  simswap_Fake/test/306_278: 32 frames
    sample path: deepfakes_detection_datasets/DF40/simswap/ff/frames/306_278/328.png

=== simswap.zip internal structure (first 8 entries) ===
  total entries: 49580
    cdf/frames/id9_id0_0006/374.png
    cdf/frames/id9_id0_0006/077.png
    cdf/frames/id9_id0_0006/180.png
    cdf/frames/id9_id0_0006/154.png
    cdf/frames/id9_id0_0006/309.png
    cdf/frames/id9_id0_0006/258.png
    cdf/frames/id9_id0_0006/270.png
    cdf/frames/id9_id0_0006/245.png


In [9]:
import glob, os, json
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# what DF40 JSONs exist? look for cdf vs ff variants
print("=== all DF40 JSONs (looking for cdf/ff variants) ===")
jsons = sorted(glob.glob(f"{REPO}/data/df40/dataset_json/*.json"))
for j in jsons:
    print("  ", os.path.basename(j))

=== all DF40 JSONs (looking for cdf/ff variants) ===
   CollabDiff.json
   DF40_all.json
   DiT_cdf.json
   DiT_ff.json
   EFSAll_cdf.json
   EFSAll_ff.json
   FRAll_cdf.json
   FRAll_ff.json
   FSAll_cdf.json
   FSAll_ff.json
   MRAA_cdf.json
   MRAA_ff.json
   MidJourney.json
   SiT_cdf.json
   SiT_ff.json
   StyleGAN2_cdf.json
   StyleGAN2_ff.json
   StyleGAN3_cdf.json
   StyleGAN3_ff.json
   StyleGANXL_cdf.json
   StyleGANXL_ff.json
   VQGAN_cdf.json
   VQGAN_ff.json
   blendface_cdf.json
   blendface_ff.json
   danet_cdf.json
   danet_ff.json
   ddim_cdf.json
   ddim_ff.json
   deepfacelab.json
   e4e_cdf.json
   e4e_ff.json
   e4s_cdf.json
   e4s_ff.json
   facedancer_cdf.json
   facedancer_ff.json
   faceswap_cdf.json
   faceswap_ff.json
   facevid2vid_cdf.json
   facevid2vid_ff.json
   fomm_cdf.json
   fomm_ff.json
   fsgan_cdf.json
   fsgan_ff.json
   heygen.json
   hyperreenact_cdf.json
   hyperreenact_ff.json
   inswap_cdf.json
   inswap_ff.json
   lia_cdf.json
   lia_ff.jso

In [10]:
import json, os, glob
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# check the CDF json (matches your cdf zip)
J = json.load(open(f"{REPO}/data/df40/dataset_json/simswap_cdf.json"))
top = list(J.keys())[0]
subs = list(J[top].keys())
print("simswap_cdf.json top:", top, "| subs:", subs)
for sub in subs:
    for split in ['test','train','val']:
        if split in J[top][sub] and J[top][sub][split]:
            vid = list(J[top][sub][split].keys())[0]
            fr = J[top][sub][split][vid]['frames']
            print(f"  {sub}/{split}/{vid}: {len(fr)} frames | sample: {fr[0]}")
            break

simswap_cdf.json top: simswap_cdf | subs: ['simswap_Real', 'simswap_Fake']
  simswap_Real/test/00170: 31 frames | sample: deepfakes_detection_datasets/Celeb-DF-v2/YouTube-real/frames/00170/055.png
  simswap_Fake/test/id9_id0_0006: 32 frames | sample: deepfakes_detection_datasets/DF40/simswap/cdf/frames/id9_id0_0006/374.png


In [11]:
import sys, importlib.util, json, os, glob, zipfile
import pandas as pd
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
method = "simswap"

# --- unzip the cdf fakes locally ---
DF40_FRAMES = "/content/df40_frames"
os.makedirs(DF40_FRAMES, exist_ok=True)
zp = f"{REPO}/data/df40_core/test/{method}.zip"
if not glob.glob(f"{DF40_FRAMES}/cdf/frames/*"):
    print(f"unzipping {method}...")
    with zipfile.ZipFile(zp) as z: z.extractall(DF40_FRAMES)
fake_frames = glob.glob(f"{DF40_FRAMES}/**/*.png", recursive=True)
print(f"unzipped {len(fake_frames)} fake frames")

# --- load data_prep, build manifest from CDF json ---
spec = importlib.util.spec_from_file_location("data_prep", f"{REPO}/src/data_prep.py")
data_prep = importlib.util.module_from_spec(spec); sys.modules["data_prep"]=data_prep
spec.loader.exec_module(data_prep)

jpath = f"{REPO}/data/df40/dataset_json/{method}_cdf.json"
df = data_prep.build_manifest_from_json(f"{method}_cdf", jpath, frames_root=None)
print("raw manifest:", len(df), "| labels:", df['label'].value_counts().to_dict())

# --- build remap indices ---
# fakes: live under DF40_FRAMES, keyed by videoid/frame.png
fake_index = {"/".join(fp.split("/")[-2:]): fp for fp in fake_frames}
# reals: your cropped Celeb-DF frames, keyed by videoid/frame.png
CDF_REAL = f"{REPO}/data/frames/Celeb-DF-v2"
real_frames = glob.glob(f"{CDF_REAL}/**/frames/**/*.png", recursive=True)
real_index = {"/".join(fp.split("/")[-2:]): fp for fp in real_frames}
print(f"fake index: {len(fake_index)} | real index: {len(real_index)}")

def remap(row):
    key = "/".join(row["frame_path"].split("/")[-2:])
    if row["label"] == 1:   # fake
        return fake_index.get(key)
    else:                   # real
        return real_index.get(key)

df["frame_path"] = df.apply(remap, axis=1)
before = len(df)
df = df[df["frame_path"].notna()].reset_index(drop=True)
print(f"after remap: {len(df)}/{before} resolved | labels: {df['label'].value_counts().to_dict()}")

df.to_parquet(f"/content/df40_{method}_manifest.parquet", index=False)
print(f"saved {method} manifest")

unzipped 24790 fake frames
raw manifest: 25942 | labels: {1: 20322, 0: 5620}
fake index: 24790 | real index: 16420
after remap: 25942/25942 resolved | labels: {1: 20322, 0: 5620}
saved simswap manifest


In [13]:
import sys
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# is there a stale 'detectors' stub?
print("detectors in sys.modules:", "detectors" in sys.modules)
if "detectors" in sys.modules:
    d = sys.modules["detectors"]
    print("  has DETECTOR:", hasattr(d, "DETECTOR"))
    if hasattr(d, "DETECTOR"):
        print("  registered:", list(d.DETECTOR.data.keys()) if hasattr(d.DETECTOR, "data") else "?")

# check the metrics.registry DETECTOR directly
from metrics.registry import DETECTOR as REG
print("metrics.registry DETECTOR keys:", list(REG.data.keys()) if hasattr(REG, "data") else "?")

detectors in sys.modules: True
  has DETECTOR: True
  registered: ['multi_attention', 'xception']
metrics.registry DETECTOR keys: []


In [14]:
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
path = f"{REPO}/src/inference.py"
src = open(path).read()

old = '''    # stub the 'detectors' package so `from detectors import DETECTOR` resolves
    # without running its __init__ (which pulls slowfast/fvcore/etc.)
    if "detectors" not in sys.modules:
        from metrics.registry import DETECTOR  # registry lives in metrics, light import
        stub = types.ModuleType("detectors")
        stub.DETECTOR = DETECTOR
        stub.__path__ = [det_dir]
        sys.modules["detectors"] = stub'''

new = '''    # stub the 'detectors' package so `from detectors import DETECTOR` resolves
    # without running its __init__ (which pulls slowfast/fvcore/etc.).
    # ALWAYS rebuild against the CURRENT metrics.registry (a stale stub from a
    # previous cell can hold a divergent registry instance -> empty-registry KeyError).
    from metrics.registry import DETECTOR
    stub = types.ModuleType("detectors")
    stub.DETECTOR = DETECTOR
    stub.__path__ = [det_dir]
    sys.modules["detectors"] = stub
    # ensure the networks package backbones are registered (populates BACKBONE);
    # import the specific backbone module the detector needs.
    import importlib as _il
    try:
        _il.import_module("networks")
    except Exception:
        # networks/__init__ may also pull heavy deps; import the xception net directly
        _bspec = importlib.util.spec_from_file_location(
            "networks.xception", f"{train_dir}/networks/xception.py")
        if _bspec is not None:
            _bmod = importlib.util.module_from_spec(_bspec)
            sys.modules.setdefault("networks", types.ModuleType("networks"))
            sys.modules["networks"].__path__ = [f"{train_dir}/networks"]
            sys.modules["networks.xception"] = _bmod
            _bspec.loader.exec_module(_bmod)'''

src = src.replace(old, new)
open(path, "w").write(src)
print("patched: stub always rebuilt + backbone registration ensured")

patched: stub always rebuilt + backbone registration ensured


In [15]:
import sys, importlib.util
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB = f"{REPO}/external/DeepfakeBench"

# full purge
for k in list(sys.modules.keys()):
    if k.startswith("detectors") or k.startswith("networks") or k=="metrics" or k.startswith("metrics.") or k=="inference":
        del sys.modules[k]
sys.path = [p for p in sys.path if p not in (f"{DFB}/training", f"{REPO}/src", DFB)]
sys.path.insert(0, DFB); sys.path.insert(0, f"{DFB}/training"); sys.path.append(f"{REPO}/src")

spec = importlib.util.spec_from_file_location("inference", f"{REPO}/src/inference.py")
inference = importlib.util.module_from_spec(spec); sys.modules["inference"]=inference
spec.loader.exec_module(inference)

model, device, info = inference.load_detector(
    dfb_root=DFB, backbone_name="xception",
    ckpt_path=f"{REPO}/weights/train_on_fs/xception.pth")
print("load:", info, "device:", device)

load: {'missing': 0, 'unexpected': 0} device: cuda


In [16]:
import torch, hashlib, os
import pandas as pd
from sklearn.metrics import roc_auc_score
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
method = "simswap"

# verify it's the FS checkpoint (not stale)
fs_path = f"{REPO}/weights/train_on_fs/xception.pth"
tag = hashlib.md5(fs_path.encode()).hexdigest()[:8]
fs_sd = torch.load(f"/content/_ckpt_xception_{tag}.pth", map_location="cpu")
mp = dict(model.named_parameters()); mk = [k for k in mp if 'conv1' in k][0]
print("model matches FS:", torch.allclose(mp[mk].cpu(), fs_sd["module.backbone.conv1.weight"]))

# score simswap
mani = pd.read_parquet(f"/content/df40_{method}_manifest.parquet")
scores = inference.score_manifest(model, device, mani, batch_size=64)
os.makedirs(f"{REPO}/reports/scores", exist_ok=True)
scores.to_parquet(f"{REPO}/reports/scores/xceptionFS_df40_{method}.parquet", index=False)

print(f"\n{method} AUC (FS xception): {roc_auc_score(scores.label, scores.prob_fake):.4f}")
print("scored", len(scores), "| labels:", scores['label'].value_counts().to_dict())

model matches FS: True
  scored 64/25942
  scored 704/25942
  scored 1344/25942
  scored 1984/25942
  scored 2624/25942
  scored 3264/25942
  scored 3904/25942
  scored 4544/25942
  scored 5184/25942
  scored 5824/25942
  scored 6464/25942
  scored 7104/25942
  scored 7744/25942
  scored 8384/25942
  scored 9024/25942
  scored 9664/25942
  scored 10304/25942
  scored 10944/25942
  scored 11584/25942
  scored 12224/25942
  scored 12864/25942
  scored 13504/25942
  scored 14144/25942
  scored 14784/25942
  scored 15424/25942
  scored 16064/25942
  scored 16704/25942
  scored 17344/25942
  scored 17984/25942
  scored 18624/25942
  scored 19264/25942
  scored 19904/25942
  scored 20544/25942
  scored 21184/25942
  scored 21824/25942
  scored 22464/25942
  scored 23104/25942
  scored 23744/25942
  scored 24384/25942
  scored 25024/25942
  scored 25664/25942

simswap AUC (FS xception): 0.9469
scored 25942 | labels: {1: 20322, 0: 5620}


In [18]:
# is the model + inference still loaded and usable?
try:
    print("model exists:", 'model' in dir() and model is not None)
    print("inference exists:", 'inference' in dir())
    print("device:", device if 'device' in dir() else "not set")
    # is DeepfakeBench on path?
    import sys
    REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
    print("DFB training on path:", f"{REPO}/external/DeepfakeBench/training" in sys.path)
except Exception as e:
    print("state check error:", e)

model exists: True
inference exists: True
device: cuda
DFB training on path: False


In [19]:
# quick test: can the already-loaded model still score?
import pandas as pd
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
test_mani = pd.read_parquet(f"/content/df40_simswap_manifest.parquet").head(64)
try:
    s = inference.score_manifest(model, device, test_mani, batch_size=64, verbose=False)
    print("scoring works! sample AUC-able rows:", len(s), "| prob range:",
          f"[{s.prob_fake.min():.3f}, {s.prob_fake.max():.3f}]")
    print("=> model is live, run Pass 1 directly")
except Exception as e:
    print("scoring failed:", type(e).__name__, str(e)[:200])
    print("=> need to reload inference env first")

scoring works! sample AUC-able rows: 64 | prob range: [0.082, 1.000]
=> model is live, run Pass 1 directly


In [17]:
import sys, importlib.util, os
import numpy as np, pandas as pd
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
method = "simswap"

# switch to calibration env (src first, purge DFB metrics)
for k in list(sys.modules.keys()):
    if k=="metrics" or k.startswith("metrics.") or k=="calibration":
        del sys.modules[k]
sys.path = [p for p in sys.path if "DeepfakeBench" not in p]
if f"{REPO}/src" in sys.path: sys.path.remove(f"{REPO}/src")
sys.path.insert(0, f"{REPO}/src")
import calibration as cal, metrics as met

scores = pd.read_parquet(f"{REPO}/reports/scores/xceptionFS_df40_{method}.parquet")
p = scores.prob_fake.values.astype(float); y = scores.label.values.astype(int)
groups = scores.identity_id.values if "identity_id" in scores.columns else None

ci, ti, info = cal.leakage_safe_split(y, groups=groups, calib_frac=0.5, seed=42)
p_cal, _ = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)
ece_raw = met.ece(p[ti], y[ti], n_bins=15, scheme="equal_mass")
ece_cal = met.ece(p_cal, y[ti], n_bins=15, scheme="equal_mass")
auc = met.roc_auc(p[ti], y[ti]) if hasattr(met,"roc_auc") else float("nan")
print(f"{method}: AUC={auc:.4f}  ECE_raw={ece_raw:.4f}  ECE_cal={ece_cal:.4f}")

# append to DF40 coupling pool
row = {"checkpoint":"train_on_fs","method":f"df40_{method}","family":"FS",
       "n":int(len(ti)),"AUC":auc,"ECE_raw":ece_raw,"ECE_cal":ece_cal}
pool_path = f"{REPO}/reports/calibration/coupling_df40.csv"
pool = pd.read_csv(pool_path) if os.path.exists(pool_path) else pd.DataFrame()
pool = pd.concat([pool, pd.DataFrame([row])]).drop_duplicates("method", keep="last").reset_index(drop=True)
pool.to_csv(pool_path, index=False)
print(f"\nDF40 coupling pool: {len(pool)} methods")
print(pool[["method","AUC","ECE_cal"]].to_string(index=False))

simswap: AUC=0.9478  ECE_raw=0.0828  ECE_cal=0.0238

DF40 coupling pool: 1 methods
      method      AUC  ECE_cal
df40_simswap 0.947799 0.023779


In [20]:
import os, glob, json, zipfile, shutil
import pandas as pd
import importlib.util, sys
from sklearn.metrics import roc_auc_score
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# data_prep loader
spec = importlib.util.spec_from_file_location("data_prep", f"{REPO}/src/data_prep.py")
data_prep = importlib.util.module_from_spec(spec); sys.modules["data_prep"]=data_prep
spec.loader.exec_module(data_prep)

# real index (Celeb-DF crop) — same for all, build once
CDF_REAL = f"{REPO}/data/frames/Celeb-DF-v2"
real_index = {"/".join(fp.split("/")[-2:]): fp
              for fp in glob.glob(f"{CDF_REAL}/**/frames/**/*.png", recursive=True)}
print(f"real index (Celeb-DF): {len(real_index)} frames\n")

methods = ["inswap","blendface","fomm","facevid2vid","StyleGAN2","sd2.1","ddim","styleclip"]

for method in methods:
    print(f"=== {method} ===")
    zp = f"{REPO}/data/df40_core/test/{method}.zip"
    jpath = f"{REPO}/data/df40/dataset_json/{method}_cdf.json"
    if not os.path.exists(zp):
        print("  zip missing, skip"); continue
    if not os.path.exists(jpath):
        print(f"  {method}_cdf.json missing, skip"); continue

    fdir = f"/content/df40_{method}"
    if not os.path.isdir(fdir):
        os.makedirs(fdir, exist_ok=True)
        with zipfile.ZipFile(zp) as z: z.extractall(fdir)
    fake_index = {"/".join(fp.split("/")[-2:]): fp
                  for fp in glob.glob(f"{fdir}/**/*.png", recursive=True)}

    try:
        df = data_prep.build_manifest_from_json(f"{method}_cdf", jpath, frames_root=None)
    except Exception as e:
        print(f"  manifest build failed: {str(e)[:120]}"); shutil.rmtree(fdir, ignore_errors=True); continue

    def remap(row):
        key = "/".join(row["frame_path"].split("/")[-2:])
        return fake_index.get(key) if row["label"]==1 else real_index.get(key)
    df["frame_path"] = df.apply(remap, axis=1)
    df = df[df["frame_path"].notna()].reset_index(drop=True)
    bal = df['label'].value_counts().to_dict()
    print(f"  manifest: {len(df)} frames, labels {bal}")
    if df['label'].nunique() < 2:
        print("  SKIP: only one label"); shutil.rmtree(fdir, ignore_errors=True); continue

    scores = inference.score_manifest(model, device, df, batch_size=64, verbose=False)
    scores.to_parquet(f"{REPO}/reports/scores/xceptionFS_df40_{method}.parquet", index=False)
    auc = roc_auc_score(scores.label, scores.prob_fake)
    print(f"  AUC = {auc:.4f}  (n={len(scores)})")
    shutil.rmtree(fdir, ignore_errors=True)

print("\n=== Pass 1 done ===")
for f in sorted(glob.glob(f"{REPO}/reports/scores/xceptionFS_df40_*.parquet")):
    print("  ", os.path.basename(f))

real index (Celeb-DF): 16420 frames

=== inswap ===
  manifest: 19053 frames, labels {1: 13433, 0: 5620}
  AUC = 0.8474  (n=19053)
=== blendface ===
  manifest: 25943 frames, labels {1: 20323, 0: 5620}
  AUC = 0.9464  (n=25943)
=== fomm ===
  manifest: 25898 frames, labels {1: 20278, 0: 5620}
  AUC = 0.8032  (n=25898)
=== facevid2vid ===
  manifest: 25510 frames, labels {1: 19890, 0: 5620}
  AUC = 0.8106  (n=25510)
=== StyleGAN2 ===
  manifest: 33794 frames, labels {1: 28174, 0: 5620}
  AUC = 0.6567  (n=33794)
=== sd2.1 ===
  manifest: 33794 frames, labels {1: 28174, 0: 5620}
  AUC = 0.6941  (n=33794)
=== ddim ===
  manifest: 33794 frames, labels {1: 28174, 0: 5620}
  AUC = 0.7708  (n=33794)
=== styleclip ===
  styleclip_cdf.json missing, skip

=== Pass 1 done ===
   xceptionFS_df40_StyleGAN2.parquet
   xceptionFS_df40_blendface.parquet
   xceptionFS_df40_ddim.parquet
   xceptionFS_df40_facevid2vid.parquet
   xceptionFS_df40_fomm.parquet
   xceptionFS_df40_inswap.parquet
   xceptionFS_

In [21]:
import sys, importlib.util, os, glob
import numpy as np, pandas as pd
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# calibration env (src first, purge DFB)
for k in list(sys.modules.keys()):
    if k=="metrics" or k.startswith("metrics.") or k=="calibration":
        del sys.modules[k]
sys.path = [p for p in sys.path if "DeepfakeBench" not in p]
if f"{REPO}/src" in sys.path: sys.path.remove(f"{REPO}/src")
sys.path.insert(0, f"{REPO}/src")
import calibration as cal, metrics as met

# family map for the methods
fam = {"simswap":"FS","inswap":"FS","blendface":"FS","fomm":"FR","facevid2vid":"FR",
       "StyleGAN2":"EFS","sd2.1":"EFS","ddim":"EFS"}

rows = []
for f in sorted(glob.glob(f"{REPO}/reports/scores/xceptionFS_df40_*.parquet")):
    method = os.path.basename(f).replace("xceptionFS_df40_","").replace(".parquet","")
    scores = pd.read_parquet(f)
    p = scores.prob_fake.values.astype(float); y = scores.label.values.astype(int)
    groups = scores.identity_id.values if "identity_id" in scores.columns else None
    ci, ti, info = cal.leakage_safe_split(y, groups=groups, calib_frac=0.5, seed=42)
    p_cal, _ = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)
    ece_raw = met.ece(p[ti], y[ti], n_bins=15, scheme="equal_mass")
    ece_cal = met.ece(p_cal, y[ti], n_bins=15, scheme="equal_mass")
    auc = met.roc_auc(p[ti], y[ti]) if hasattr(met,"roc_auc") else float("nan")
    rows.append({"method":f"df40_{method}","family":fam.get(method,"?"),
                 "n":int(len(ti)),"AUC":auc,"ECE_raw":ece_raw,"ECE_cal":ece_cal})
    print(f"  {method:14s} [{fam.get(method,'?'):3s}] AUC={auc:.4f}  ECE_cal={ece_cal:.4f}")

pool = pd.DataFrame(rows)
pool.to_csv(f"{REPO}/reports/calibration/coupling_df40.csv", index=False)

from scipy.stats import pearsonr, spearmanr
r_p, p_p = pearsonr(pool.AUC, pool.ECE_cal)
r_s, p_s = spearmanr(pool.AUC, pool.ECE_cal)
print(f"\n=== DF40 coupling ({len(pool)} methods) ===")
print(f"AUC vs ECE_cal: Pearson r={r_p:.3f} (p={p_p:.4f}) | Spearman ρ={r_s:.3f} (p={p_s:.4f})")

  StyleGAN2      [EFS] AUC=0.6491  ECE_cal=0.0629
  blendface      [FS ] AUC=0.9429  ECE_cal=0.0225
  ddim           [EFS] AUC=0.7661  ECE_cal=0.0411
  facevid2vid    [FR ] AUC=0.8216  ECE_cal=0.0637
  fomm           [FR ] AUC=0.8008  ECE_cal=0.0544
  inswap         [FS ] AUC=0.8152  ECE_cal=0.0726
  sd2.1          [EFS] AUC=0.6879  ECE_cal=0.0442
  simswap        [FS ] AUC=0.9478  ECE_cal=0.0238

=== DF40 coupling (8 methods) ===
AUC vs ECE_cal: Pearson r=-0.585 (p=0.1277) | Spearman ρ=-0.333 (p=0.4198)


In [22]:
import pandas as pd, numpy as np, os
from scipy.stats import pearsonr, spearmanr
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

ffpp = pd.read_csv(f"{REPO}/reports/calibration/coupling_pooled_2ckpt.csv")
df40 = pd.read_csv(f"{REPO}/reports/calibration/coupling_df40.csv")
df40["checkpoint"] = "train_on_fs"
# unify columns
ffpp_u = ffpp[["method","AUC","ECE_cal"]].copy(); ffpp_u["source"]="FFpp"
df40_u = df40[["method","AUC","ECE_cal"]].copy(); df40_u["source"]="DF40"
allpts = pd.concat([ffpp_u, df40_u]).reset_index(drop=True)

print(f"=== combined {len(allpts)} points (FF++ {len(ffpp_u)} + DF40 {len(df40_u)}) ===")
print(allpts.sort_values("AUC").to_string(index=False))

r_p, p_p = pearsonr(allpts.AUC, allpts.ECE_cal)
r_s, p_s = spearmanr(allpts.AUC, allpts.ECE_cal)
print(f"\nFULL-RANGE coupling: Pearson r={r_p:.3f} (p={p_p:.4f}) | Spearman ρ={r_s:.3f} (p={p_s:.4f})")

allpts.to_csv(f"{REPO}/reports/calibration/coupling_all_16pts.csv", index=False)

=== combined 16 points (FF++ 8 + DF40 8) ===
          method      AUC  ECE_cal source
        faceswap 0.386802 0.326051   FFpp
       face2face 0.494803 0.262362   FFpp
  neuraltextures 0.600081 0.342808   FFpp
       face2face 0.625260 0.319473   FFpp
  neuraltextures 0.640339 0.158202   FFpp
  df40_StyleGAN2 0.649117 0.062928   DF40
       deepfakes 0.664467 0.201341   FFpp
      df40_sd2.1 0.687866 0.044155   DF40
       df40_ddim 0.766139 0.041084   DF40
       df40_fomm 0.800843 0.054372   DF40
     df40_inswap 0.815188 0.072649   DF40
df40_facevid2vid 0.821552 0.063689   DF40
       deepfakes 0.839122 0.176211   FFpp
  df40_blendface 0.942852 0.022461   DF40
    df40_simswap 0.947799 0.023779   DF40
        faceswap 0.973808 0.036445   FFpp

FULL-RANGE coupling: Pearson r=-0.779 (p=0.0004) | Spearman ρ=-0.791 (p=0.0003)


In [23]:
import os, subprocess, json
from datetime import datetime
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
os.chdir(REPO)

# update findings JSON with the combined result
findings = json.load(open(f"{REPO}/reports/calibration/NB02_findings.json"))
findings["df40_scaling"] = {
    "date": datetime.now().isoformat(),
    "df40_methods_scored": 8,
    "df40_only_coupling": {"pearson_r": -0.585, "pearson_p": 0.128, "note": "weak/ns - DF40 calibrates well across competence range"},
    "combined_16pt_coupling": {"pearson_r": -0.779, "pearson_p": 0.0004, "spearman_rho": -0.791, "spearman_p": 0.0003},
    "interpretation": "coupling significant across full AUC range (0.39-0.97); driven by low-competence regime; DF40 high-competence methods calibrate well regardless of AUC; possible dataset effect (FFpp higher ECE than DF40 at similar AUC) to disentangle"
}
json.dump(findings, open(f"{REPO}/reports/calibration/NB02_findings.json","w"), indent=2)

# restore git identity
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"/content/drive/MyDrive/CDTS_Research/{f}"):
        subprocess.run(f'cp "/content/drive/MyDrive/CDTS_Research/{f}" /root/{f}', shell=True)

# stage DF40 results
subprocess.run("git add reports/scores/xceptionFS_df40_*.parquet reports/calibration/coupling_df40.csv reports/calibration/coupling_all_16pts.csv reports/calibration/NB02_findings.json src/inference.py", shell=True)
r = subprocess.run("git status --short", shell=True, capture_output=True, text=True)
print(r.stdout)

 m external/DeepfakeBench
 M notebooks/GBDF_download.ipynb
M  reports/calibration/NB02_findings.json
A  reports/calibration/coupling_all_16pts.csv
A  reports/calibration/coupling_df40.csv
A  reports/scores/xceptionFS_df40_StyleGAN2.parquet
A  reports/scores/xceptionFS_df40_blendface.parquet
A  reports/scores/xceptionFS_df40_ddim.parquet
A  reports/scores/xceptionFS_df40_facevid2vid.parquet
A  reports/scores/xceptionFS_df40_fomm.parquet
A  reports/scores/xceptionFS_df40_inswap.parquet
A  reports/scores/xceptionFS_df40_sd2.1.parquet
A  reports/scores/xceptionFS_df40_simswap.parquet
M  src/inference.py
?? notebooks/NB03_df40_scaling.ipynb



## Cell 7 — Calibrate this method + extend the coupling

Compute ECE_cal for this method (calibration fit leakage-safe), append its (AUC, ECE_cal)
to the running coupling table, recompute the correlation.


In [ ]:
import sys, importlib.util, os
import numpy as np, pandas as pd
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
method = "simswap"

# switch to calibration env (src first, purge DFB metrics)
for k in list(sys.modules.keys()):
    if k=="metrics" or k.startswith("metrics.") or k in ("calibration","metrics"):
        del sys.modules[k]
sys.path = [p for p in sys.path if "DeepfakeBench" not in p]
if f"{REPO}/src" in sys.path: sys.path.remove(f"{REPO}/src")
sys.path.insert(0, f"{REPO}/src")
import calibration as cal, metrics as met

scores = pd.read_parquet(f"{REPO}/reports/scores/xceptionFS_df40_{method}.parquet")
p = scores.prob_fake.values.astype(float); y = scores.label.values.astype(int)
groups = scores.identity_id.values if "identity_id" in scores.columns else None

# leakage-safe split, fit hybrid, measure ECE before/after
ci, ti, info = cal.leakage_safe_split(y, groups=groups, calib_frac=0.5, seed=42)
p_cal, cal_obj = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)
ece_raw = met.ece(p[ti], y[ti], n_bins=15, scheme="equal_mass")
ece_cal = met.ece(p_cal, y[ti], n_bins=15, scheme="equal_mass")
auc = met.roc_auc(p[ti], y[ti]) if hasattr(met,"roc_auc") else float("nan")
print(f"{method}: AUC={auc:.4f}  ECE_raw={ece_raw:.4f}  ECE_cal={ece_cal:.4f}")

# append to the coupling pool
row = {"checkpoint":"train_on_fs","method":f"df40_{method}","family":"FS",
       "n":int(len(ti)),"AUC":auc,"ECE_raw":ece_raw,"ECE_cal":ece_cal}
pool_path = f"{REPO}/reports/calibration/coupling_df40.csv"
pool = pd.read_csv(pool_path) if os.path.exists(pool_path) else pd.DataFrame()
pool = pd.concat([pool, pd.DataFrame([row])]).drop_duplicates("method", keep="last").reset_index(drop=True)
pool.to_csv(pool_path, index=False)
print(f"\ncoupling pool now {len(pool)} DF40 methods")
print(pool[["method","AUC","ECE_cal"]].to_string(index=False))

## Cell 8 — (after one method verified) loop over all local core methods

Once the single-method flow works end-to-end, this loops the same steps over all 9 local
core methods, building a 9-point coupling table. Then recompute correlation across all points.


In [ ]:
# Placeholder — fill in after Cells 2-7 verified on simswap.
# The loop wraps: unzip -> manifest+remap -> (add reals) -> score -> calibrate -> append.
# Methods: simswap, inswap, blendface (FS); fomm, facevid2vid (FR);
#          StyleGAN2, sd2.1, ddim (EFS); styleclip (FE)
print("run after single-method flow is confirmed working")